#Complete Data Ingestion and Preprocessing



---

## PART 1: INSTALLATION AND SETUP

Install all required dependencies. Run this cell first.

In [ ]:
import torch
torch.cuda.is_available()

True

In [ ]:

!pip -q install -U pip

# Pin ML stack (works with typical 298B TRL notebooks)
!pip -q install "transformers==4.46.2" "trl==0.12.2" "accelerate==0.34.2" "peft==0.13.2"

# Pin numpy/pandas to avoid Colab conflicts
!pip -q install "numpy==2.0.2" "pandas==2.2.2"

# Other deps (avoid -U here to prevent pulling incompatible upgrades)
!pip -q install datasets scikit-learn tqdm beautifulsoup4 lxml html5lib google-generativeai kagglehub

# Optional: flash-attn (skip if it builds; building causes hangs)
print("Installation complete ✅")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 74.1 MB/s eta 0:00:00
Installation complete ✅


In [ ]:
import numpy as np, pandas as pd
import transformers, trl
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("transformers", transformers.__version__)
print("trl", trl.__version__)

numpy 2.0.2
pandas 2.2.2
transformers 4.46.2
trl 0.12.2


In [ ]:
#!pip -q install -U google-genai

## Import Required Libraries

In [ ]:
# =========================
# Standard library imports
# =========================
import os
import sys
import json
import re
import hashlib
import logging
import warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional

# =========================
# Data processing
# =========================
import pandas as pd
import numpy as np
from tqdm import tqdm

# =========================
# Email parsing
# =========================
from email import policy
from email.parser import Parser
from email.header import decode_header
from email.utils import parsedate_to_datetime, getaddresses
from bs4 import BeautifulSoup

# =========================
# Machine learning
# =========================
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support

# =========================
# Hugging Face auth
# =========================
from huggingface_hub import login

# =========================
# Transformers (NO bitsandbytes)
# =========================
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    pipeline
)

# =========================
# PEFT (LoRA only, no QLoRA)
# =========================
from peft import LoraConfig, get_peft_model, PeftModel

# =========================
# TRL
# =========================
from trl import SFTTrainer

# =========================
# Gemini API
# =========================
import google.generativeai as genai

warnings.filterwarnings("ignore")

print("All libraries imported successfully ✅")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

All libraries imported successfully ✅
PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA device: NVIDIA A100-SXM4-40GB


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## Configure API Keys and Settings

**IMPORTANT:** Set your API keys here.

**Required API Keys:**
1. **GEMINI_API_KEY**: Get from https://makersuite.google.com/app/apikey
2. **HF_TOKEN**: Get from https://huggingface.co/settings/tokens (needed for Llama models)

**Alternatively, use Google Colab Secrets:**
- Go to the key icon on the left sidebar
- Add secrets: `GEMINI_API_KEY` and `HF_TOKEN`
- Enable notebook access

In [ ]:
import google.generativeai as genai

# Set your key here temporarily (or use Config.GEMINI_API_KEY)
genai.configure(api_key="")

try:
    model = genai.GenerativeModel("gemini-2.5-flash")
    response = model.generate_content("Say 'API working' if you can read this.")
    print("✅ Gemini Response:", response.text)

except Exception as e:
    print("❌ Gemini Error:")
    print(e)

✅ Gemini Response: API working


In [ ]:
# Configuration for the entire pipeline
# Modified for FP16 LoRA (no bitsandbytes)

class Config:
    """Global configuration for preprocessing and fine-tuning."""

    # ==================================================
    # API KEYS
    # ==================================================
    GEMINI_API_KEY = ""
    HF_TOKEN = ""

    # ==================================================
    # SAMPLE SIZE
    # ==================================================
    SAMPLE_SIZE = 500

    # ==================================================
    # PATHS
    # ==================================================
    DATA_DIR = "./data"
    OUTPUT_DIR = "./output"
    CHECKPOINT_DIR = "./checkpoints"
    MODEL_DIR = "./trained_model"
    FINETUNING_OUTPUT_DIR = "./finetuning_output"

    # ==================================================
    # PREPROCESSING SETTINGS
    # ==================================================
    MIN_BODY_LENGTH = 50
    MAX_BODY_LENGTH = 5000
    MEETING_CONFIDENCE_THRESHOLD = 0.5

    USE_DEDUPLICATION = True
    USE_ZERO_SHOT_FILTER = True
    USE_CHECKPOINTING = True

    # ==================================================
    # FINE-TUNING SETTINGS
    # ==================================================
    BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

    TRAIN_RATIO = 0.90
    VAL_RATIO = 0.05
    TEST_RATIO = 0.05

    NUM_EPOCHS = 3
    BATCH_SIZE = 4
    GRADIENT_ACCUMULATION_STEPS = 4
    LEARNING_RATE = 2e-4
    MAX_SEQ_LENGTH = 2048
    WARMUP_RATIO = 0.1

    # LoRA configuration
    LORA_R = 16
    LORA_ALPHA = 32
    LORA_DROPOUT = 0.05
    TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

    # 🔥 IMPORTANT: Disable 4-bit quantization
    USE_4BIT_QUANTIZATION = False

    # Logging
    LOGGING_STEPS = 10
    EVAL_STEPS = 50
    SAVE_STEPS = 50

    @classmethod
    def setup(cls):
        os.makedirs(cls.DATA_DIR, exist_ok=True)
        os.makedirs(cls.OUTPUT_DIR, exist_ok=True)
        os.makedirs(cls.CHECKPOINT_DIR, exist_ok=True)
        os.makedirs(cls.MODEL_DIR, exist_ok=True)
        os.makedirs(cls.FINETUNING_OUTPUT_DIR, exist_ok=True)
        print("Directories created successfully")

    @classmethod
    def validate(cls):
        errors = []

        if not cls.GEMINI_API_KEY:
            errors.append("GEMINI_API_KEY not set")

        if not cls.HF_TOKEN:
            errors.append("HF_TOKEN not set")

        if errors:
            print("WARNING: Missing API keys:")
            for error in errors:
                print(f"  - {error}")
            return False

        return True


# Create directories
Config.setup()

# Validate API keys
if Config.validate():
    print("\nConfiguration complete ✅")
    print(f"Sample size: {Config.SAMPLE_SIZE}")
    print(f"Base model: {Config.BASE_MODEL}")

Directories created successfully

Configuration complete ✅
Sample size: 500
Base model: meta-llama/Llama-3.2-3B-Instruct


## Authenticate with Hugging Face

This cell authenticates with Hugging Face to access gated models like Llama.

**Note:** You must accept the Llama license at https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct

In [ ]:
# Authenticate with Hugging Face
if Config.HF_TOKEN != "YOUR_HUGGINGFACE_TOKEN_HERE":
    try:
        login(token=Config.HF_TOKEN, add_to_git_credential=False)
        print("Successfully authenticated with Hugging Face")
    except Exception as e:
        print(f"Failed to authenticate with Hugging Face: {e}")
        print("Please check your HF_TOKEN and ensure you've accepted the Llama license")
else:
    print("HF_TOKEN not set. Please set it in the Config class above.")

Successfully authenticated with Hugging Face


---

## PART 2: EMAIL PREPROCESSING FUNCTIONS

These functions handle parsing, cleaning, and processing raw emails.

### Define Regular Expression Patterns

Patterns for cleaning email content (signatures, quoted text, noise).

In [ ]:
# Regex patterns for email cleaning

# Remove "Re:", "Fwd:" prefixes from subject lines
SUBJ_PREFIX = re.compile(r"^\s*((re|fwd|fw)\s*:\s*)+", re.I)

# Split email at quoted/forwarded message markers
QUOTE_SPLIT_RE = re.compile(
    r"\n\s*-{5}Original Message-{5}|\n\s*On .*? wrote:|\n\s*From:\s*.*\n",
    re.IGNORECASE | re.DOTALL
)

# Remove signatures
SIGNATURE_RE = re.compile(
    r"(?s)"
    r"("
    r"\n--\s*|\n__\s*|\n\s*(Best|Kind|Warm)\s*Regards,?"
    r"|\n\s*(Sincerely|Thanks|Cheers),?"
    r"|\n\s*This e-mail and any attachments.*"
    r"|\n\s*CONFIDENTIALITY(?: NOTICE| WARNING).*"
    r")"
    r".*$"
)

# Enron-specific noise patterns
ENCODING_ARTIFACTS = re.compile(r"=20|=3D|=09")
EXCESSIVE_DASHES = re.compile(r"^[-_]{5,}$", re.M)
MOBILE_SIGNATURES = re.compile(r"Sent from my (iPhone|BlackBerry|Android|Mobile).*$", re.I)
CONFIDENTIALITY = re.compile(
    r'This (email|message) (is|may be) confidential.*?(?=\n\n|\Z)',
    re.I | re.S
)

CALENDAR_BOILERPLATE = re.compile(
    r'(To accept|decline|tentatively accept this meeting).*?(?=\n\n|\Z)',
    re.I | re.S
)

MOJIBAKE = re.compile(r'[?]{3,}')

# Meeting-related keywords (TIGHTENED — removed "discuss")
MEETING_KEYWORDS = re.compile(
    r'\b(meet|meeting|sync|catch\s*up|call|schedule|invite|reschedul\w*|calendar)\b',
    re.I
)

print("Regular expression patterns defined")

Regular expression patterns defined


### Email Parsing Functions

In [ ]:
def _dec(h: str) -> str:
    """
    Decode email header with proper character encoding handling.

    Args:
        h: Raw header string

    Returns:
        Decoded string
    """
    if not h:
        return ""
    parts = []
    for frag, enc in decode_header(h):
        if isinstance(frag, bytes):
            try:
                parts.append(frag.decode(enc or "utf-8", "ignore"))
            except:
                parts.append(frag.decode("utf-8", "ignore"))
        else:
            parts.append(str(frag))
    return "".join(parts)


def _canon_subject(s: str) -> str:
    """
    Canonicalize subject line: remove Re:/Fwd:, normalize whitespace, lowercase.

    Args:
        s: Raw subject string

    Returns:
        Cleaned subject string
    """
    s = _dec(s or "")
    s = SUBJ_PREFIX.sub("", s).strip().lower()
    return re.sub(r"\s+", " ", s)


def _addr_list(h: str) -> List[str]:
    """
    Extract email addresses from header field.

    Args:
        h: Header field (To, Cc, etc.)

    Returns:
        List of email addresses
    """
    return [e.lower() for _, e in getaddresses([_dec(h or "")]) if e]


def _html2txt(html: str) -> str:
    """
    Convert HTML to plain text with robust fallback.
    FIXED: Tries multiple parsers to handle different environments.

    Args:
        html: HTML content

    Returns:
        Plain text
    """
    if not html:
        return ""

    # Try parsers in order of preference
    parsers = ['lxml', 'html.parser', 'html5lib']

    for parser in parsers:
        try:
            return BeautifulSoup(html, parser).get_text("\n", strip=True)
        except:
            continue

    # Fallback: regex-based HTML stripping
    text = re.sub(r'<[^>]+>', '', html)
    return text.strip()


def _body(msg) -> str:
    """
    Extract body text from email message object.
    Handles both plain text and HTML emails.

    Args:
        msg: Email message object

    Returns:
        Body text
    """
    try:
        if msg.is_multipart():
            plain, html = None, None
            for p in msg.walk():
                ct = p.get_content_type()
                if ct == "text/plain" and plain is None:
                    try:
                        plain = p.get_content()
                    except:
                        payload = p.get_payload(decode=True) or b""
                        plain = payload.decode(p.get_content_charset() or "utf-8", "ignore")
                elif ct == "text/html" and html is None:
                    try:
                        html = p.get_content()
                    except:
                        payload = p.get_payload(decode=True) or b""
                        html = payload.decode(p.get_content_charset() or "utf-8", "ignore")

            # Prefer plain text, fallback to HTML
            if plain and plain.strip():
                return plain.strip()
            if html:
                return _html2txt(html)
            return ""

        content = msg.get_content()
        if msg.get_content_type() == "text/html":
            return _html2txt(content)
        return (content or "").strip()
    except Exception as e:
        return ""


def _clean_body_advanced(text: str) -> str:
    """
    Clean email body by removing quoted replies and signatures.

    Args:
        text: Raw email body

    Returns:
        Cleaned body text
    """
    if not text:
        return ""
    # Remove quoted text
    text = QUOTE_SPLIT_RE.split(text, 1)[0]
    # Remove signatures
    text = SIGNATURE_RE.sub("", text)
    # Normalize excessive newlines
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def remove_enron_noise(text: str) -> str:
    """
    Remove Enron-specific artifacts and noise.

    Args:
        text: Email body text

    Returns:
        Cleaned text
    """
    if not text:
        return ""

    # Remove encoding artifacts
    text = ENCODING_ARTIFACTS.sub(' ', text)
    text = EXCESSIVE_DASHES.sub('', text)
    text = MOBILE_SIGNATURES.sub('', text)
    text = CONFIDENTIALITY.sub('', text)
    text = CALENDAR_BOILERPLATE.sub('', text)
    text = MOJIBAKE.sub('', text)

    # Normalize whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)

    return text.strip()


def parse_email(raw_text: str) -> Dict:
    """
    Parse raw email text into structured format.
    Main parsing function that orchestrates all cleaning steps.

    Args:
        raw_text: Raw email content in MIME format

    Returns:
        Dictionary with parsed and cleaned email components
    """
    try:
        # Parse MIME format
        msg = Parser(policy=policy.default).parsestr(raw_text or "")

        # Extract and clean body
        body_full = _body(msg)
        body_clean = _clean_body_advanced(body_full)
        body_clean = remove_enron_noise(body_clean)

        # Parse date with timezone
        dt = parsedate_to_datetime(_dec(msg.get("Date"))) if msg.get("Date") else None
        tzoff = int(dt.utcoffset().total_seconds() // 60) if (dt and dt.utcoffset()) else None

        return {
            "msg_id": _dec(msg.get("Message-ID")) or None,
            "date_utc": pd.to_datetime(dt, utc=True, errors="coerce"),
            "tz_offset_minutes": tzoff,
            "from_email": (_addr_list(msg.get("From")) or [None])[0],
            "to_emails": _addr_list(msg.get("To")),
            "cc_emails": _addr_list(msg.get("Cc")),
            "bcc_emails": _addr_list(msg.get("Bcc")),
            "subject_raw": _dec(msg.get("Subject")) or "",
            "subject_clean": _canon_subject(msg.get("Subject")),
            "body_full": body_full,
            "body_clean": body_clean,
            "parse_error": None
        }
    except Exception as e:
        # Return empty structure on parse error
        return {
            "msg_id": None,
            "date_utc": pd.NaT,
            "tz_offset_minutes": None,
            "from_email": None,
            "to_emails": [],
            "cc_emails": [],
            "bcc_emails": [],
            "subject_raw": "",
            "subject_clean": "",
            "body_full": "",
            "body_clean": "",
            "parse_error": str(e)
        }

print("Email parsing functions defined")

Email parsing functions defined


### Feature Extraction and Quality Filtering

In [ ]:
#Strip forwarded/original header lines so they don't trigger date/time features ---
HEADER_BLOCK_RE = re.compile(
    r'(?im)^(from|sent|to|cc|bcc|subject|forwarded by)\s*:\s.*$|^-----original message-----$'
)

def strip_header_blocks(text: str) -> str:
    if not text:
        return ""
    return "\n".join(
        ln for ln in text.splitlines()
        if not HEADER_BLOCK_RE.search(ln.strip())
    )


def extract_meeting_features(subject: str, body: str) -> Dict:
    """
    Extract features that indicate meeting intent.
    Used for both heuristic filtering and as input features.
    """
    body_no_headers = strip_header_blocks(body)
    text = (subject + " " + body_no_headers).lower()

    features = {
        # Temporal features (slightly improved time pattern)
        'has_time': bool(re.search(r'\b\d{1,2}(:\d{2})?\s*(am|pm)\b', text, re.I)),
        'has_date': bool(re.search(r'\d{1,2}/\d{1,2}(/\d{2,4})?', text)),
        'has_day_reference': bool(re.search(
            r'\b(monday|tuesday|wednesday|thursday|friday|saturday|sunday|tomorrow|today|next week|this week)\b',
            text
        )),

        # Meeting keywords
        'has_meeting_word': bool(MEETING_KEYWORDS.search(text)),

        # Location indicators
        'has_location': bool(re.search(
            r'\b(room|conference|office|zoom|teams|skype|call-in|building|floor)\b',
            text
        )),

        # Action/invitation words
        'has_invitation': bool(re.search(
            r'\b(join|attend|confirm|rsvp|let.?s|please|invite)\b',
            text
        )),

        # Subject indicators
        'subject_has_meeting': 'meeting' in (subject or '').lower(),

        # Length features
        'body_length': len(body),
        'word_count': len(re.findall(r'\b\w+\b', body)),
    }

    # Calculate heuristic meeting score (weighted sum)
    features['heuristic_score'] = sum([
        features['has_time'] * 0.3,
        features['has_date'] * 0.2,
        features['has_day_reference'] * 0.15,
        features['has_meeting_word'] * 0.25,
        features['has_location'] * 0.15,
        features['has_invitation'] * 0.1,
        features['subject_has_meeting'] * 0.2,
    ])

    return features


def is_quality_email(email_dict: Dict) -> Tuple[bool, str]:
    """
    Check if email meets quality standards.
    Filters out junk, auto-generated, and malformed emails.
    """
    body = email_dict.get('body_clean', '')

    # Length checks
    if len(body) < Config.MIN_BODY_LENGTH:
        return False, "too_short"
    if len(body) > Config.MAX_BODY_LENGTH:
        return False, "too_long"

    # Parse error check
    if email_dict.get('parse_error'):
        return False, "parse_error"

    # Encoding quality
    if MOJIBAKE.search(body):
        return False, "encoding_error"

    # Auto-generated email detection
    auto_generated_patterns = [
        r"This is an automatically generated",
        r"Do not reply to this",
        r"Out of office",
        r"Vacation notice",
        r"Automatic reply",
    ]
    if any(re.search(p, body, re.I) for p in auto_generated_patterns):
        return False, "auto_generated"

    # Spam indicators
    if body.count('http') > 5:
        return False, "too_many_links"

    # Substantive content check
    words = re.findall(r'\b\w+\b', body)
    if len(words) < 10:
        return False, "insufficient_content"

    # Valid sender check
    if not email_dict.get('from_email'):
        return False, "no_sender"

    return True, "pass"


def generate_content_hash(body: str) -> str:
    """
    Generate MD5 hash for deduplication.
    """
    normalized = re.sub(r'\s+', ' ', (body or '').lower()).strip()
    return hashlib.md5(normalized.encode()).hexdigest()

print("Feature extraction and filtering functions defined")

Feature extraction and filtering functions defined


---

## PART 3: ZERO-SHOT CLASSIFICATION

Uses a pre-trained model to classify emails as meeting-related or not.

**FIXED:** Bug where "not a meeting" label was incorrectly counted as positive.

In [ ]:
class MeetingClassifier:
    """
    Zero-shot classifier for meeting intent detection.

    Correct approach:
    Use a binary contrast: "meeting/scheduling" vs "not a meeting",
    then threshold the meeting score.
    """

    def __init__(self):
        print("Loading zero-shot classification model...")
        print("This may take 1-2 minutes on first run (downloading model)")

        try:
            self.classifier = pipeline(
                "zero-shot-classification",
                model="facebook/bart-large-mnli",
                device=0 if torch.cuda.is_available() else -1
            )

            # ✅ Binary labels (prevents forced meeting classification)
            self.meeting_label = "meeting or scheduling email"
            self.nonmeeting_label = "not a meeting email (report, update, FYI, spreadsheet, discussion)"
            self.candidate_labels = [self.meeting_label, self.nonmeeting_label]

            print("Zero-shot classifier loaded successfully!")

        except Exception as e:
            print(f"Failed to load zero-shot classifier: {e}")
            raise

    def classify(self, subject: str, body: str) -> Tuple[bool, float]:
        try:
            text = f"{subject} {body[:300]}"

            result = self.classifier(
                text,
                self.candidate_labels,
                multi_label=False
            )

            # Extract meeting score directly
            labels = result["labels"]
            scores = result["scores"]
            meeting_score = scores[labels.index(self.meeting_label)]

            is_meeting = meeting_score > Config.MEETING_CONFIDENCE_THRESHOLD
            return is_meeting, float(meeting_score)

        except Exception as e:
            print(f"Error in classification: {e}")
            return False, 0.0

print("Zero-shot classification class defined")

Zero-shot classification class defined


---

## PART 4: DATA LOADING AND FILTERING

Download Enron dataset and filter for meeting emails.

**Note:** The following cells will download and process the Enron dataset. This may take 15-30 minutes.

### Download Enron Dataset

In [ ]:
def download_enron_dataset() -> Optional[str]:
    """
    Download Enron dataset using kagglehub.

    Returns:
        Path to downloaded dataset or None if failed
    """
    print("Downloading Enron dataset...")
    print("This may take 5-10 minutes (downloading ~400MB)")

    try:
        import kagglehub
        path = kagglehub.dataset_download("wcukierski/enron-email-dataset")
        print(f"Dataset downloaded successfully to: {path}")
        return path
    except Exception as e:
        print(f"Error downloading dataset: {e}")
        print("")
        print("Alternative: Download manually from:")
        print("https://www.kaggle.com/datasets/wcukierski/enron-email-dataset")
        return None

# Download the dataset
enron_path = download_enron_dataset()

This may take 5-10 minutes (downloading ~400MB)
Using Colab cache for faster access to the 'enron-email-dataset' dataset.
Dataset downloaded successfully to: /kaggle/input/enron-email-dataset


In [ ]:
from pathlib import Path

p = Path(enron_path)
print("enron_path:", p)
print("Top-level files/folders:")

for x in list(p.iterdir())[:30]:
    print("-", x.name)

enron_path: /kaggle/input/enron-email-dataset
Top-level files/folders:
- emails.csv


### Load and Parse Emails

In [ ]:
import pandas as pd
from pathlib import Path

def load_enron_emails(enron_path: str, max_emails: int = 500000):
    print(f"Loading CSV emails from {enron_path}...")

    csv_path = Path(enron_path) / "emails.csv"

    df = pd.read_csv(csv_path)
    print("Total emails in dataset:", len(df))

    if max_emails:
        df = df.head(max_emails)
        print(f"Using first {len(df)} emails")

    emails = []

    for raw_text in df["message"]:
        parsed = parse_email(raw_text)
        emails.append(parsed)

    print(f"Successfully parsed {len(emails)} emails")
    return emails

# Load emails
if enron_path:
    all_emails = load_enron_emails(enron_path, max_emails=500000)
else:
    print("ERROR: Cannot proceed without dataset")
    all_emails = []

Loading CSV emails from /kaggle/input/enron-email-dataset...
Total emails in dataset: 517401
Using first 500000 emails
Successfully parsed 500000 emails


In [ ]:
import pandas as pd
import numpy as np
import random

df_all = pd.DataFrame(all_emails)

print("=== Dataset / Parsing Summary ===")
print("Total parsed emails:", len(df_all))
print("Parse errors:", df_all["parse_error"].notna().sum())

# Basic text stats
df_all["body_len"] = df_all["body_clean"].fillna("").str.len()
print("\nBody length (cleaned) stats:")
print(df_all["body_len"].describe(percentiles=[.5, .9, .95, .99]))

# Show top senders
print("\nTop 10 senders:")
display(df_all["from_email"].value_counts().head(10))

# Show 1 example: raw vs cleaned
idx = random.randint(0, len(df_all)-1)
print("\n=== Example Parsed Email ===")
print("From:", df_all.loc[idx, "from_email"])
print("Subject raw:", df_all.loc[idx, "subject_raw"])
print("Subject clean:", df_all.loc[idx, "subject_clean"])
print("\nBody clean preview:\n", (df_all.loc[idx, "body_clean"] or "")[:800])

=== Dataset / Parsing Summary ===
Total parsed emails: 500000
Parse errors: 0

Body length (cleaned) stats:
count    5.000000e+05
mean     7.949592e+02
std      3.584358e+03
min      0.000000e+00
50%      2.630000e+02
90%      1.644000e+03
95%      2.829000e+03
99%      9.080000e+03
max      1.403265e+06
Name: body_len, dtype: float64

Top 10 senders:


,count
from_email,
kay.mann@enron.com,16735
vince.kaminski@enron.com,14354
jeff.dasovich@enron.com,11405
chris.germany@enron.com,8801
sara.shackleton@enron.com,8777
tana.jones@enron.com,8484
enron.announcements@enron.com,8217
pete.davis@enron.com,7754
steven.kean@enron.com,6752



=== Example Parsed Email ===
From: lorna.brennan@enron.com
Subject raw: Changes in Registration for CERA Access
Subject clean: changes in registration for cera access

Body clean preview:
 CERA notified us yesterday of new procedures for accessing the CERA web site. 

If you have never registered for www.cera.com, please follow the procedures attached.
Call me if you have any questions.

Lorna M. Brennan
Competitive Intelligence
Enron Transportation Services
Omaha 402-398-7573


In [ ]:
print("Emails with empty cleaned body:",
      (df_all["body_len"] == 0).sum())

Emails with empty cleaned body: 825


### Initialize Zero-Shot Classifier and Filter Emails

In [ ]:
# Initialize classifier if enabled
if Config.USE_ZERO_SHOT_FILTER and len(all_emails) > 0:
    classifier = MeetingClassifier()
else:
    classifier = None
    print("Zero-shot filtering disabled or no emails to process")

Loading zero-shot classification model...
This may take 1-2 minutes on first run (downloading model)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Zero-shot classifier loaded successfully!


In [ ]:
from collections import Counter
from tqdm import tqdm

reasons = Counter()
sample_n = 20000

for e in tqdm(all_emails[:sample_n], desc="Auditing quality filter"):
    ok, reason = is_quality_email(e)
    if not ok:
        reasons[reason] += 1

print("=== Quality Filter Rejection Reasons (sample) ===")
display(pd.DataFrame(reasons.most_common(), columns=["reason", "count"]))

Auditing quality filter: 100%|██████████| 20000/20000 [00:01<00:00, 17324.94it/s]

=== Quality Filter Rejection Reasons (sample) ===


,reason,count
0,too_short,3243
1,too_long,723
2,insufficient_content,270
3,too_many_links,228
4,auto_generated,122


In [ ]:
import re
from typing import Dict, List, Tuple
from tqdm import tqdm
import torch
from transformers import pipeline

# Matches common header-ish lines including "X on 10/23/2000 11:26:59 AM"
HEADERISH_RE = re.compile(
    r'(?im)^('
    r'(from|sent|to|cc|bcc|subject)\s*:.*'
    r'|forwarded by .*'
    r'|.*\son\s+\d{1,2}/\d{1,2}/\d{2,4}\s+\d{1,2}:\d{2}:\d{2}\s*(am|pm)?\s*$'
    r'|^-+\s*original message\s*-+$'
    r')$'
)

# Business intent gate
BUSINESS_INTENT_RE = re.compile(
    r'(?i)\b('
    r'agenda|minutes|conference call|dial[- ]?in|call[- ]?in|passcode|access code|'
    r'staff meeting|team meeting|all hands|meeting|reschedul\w*|calendar invite|'
    r'conference room|room\s*[A-Z]?\d+|when\s*:|where\s*:|location\s*:'
    r')\b'
)

# Marketing/promo exclusion
MARKETING_RE = re.compile(
    r"(?i)\b("
    r"unsubscribe|click here|earnings\.com|promotion|tune in|chance to win|experience package|"
    r"view today'?s earnings|after market|earnings announcement"
    r")\b"
)

# Non-business schedule exclusions
SCHEDULE_EXCLUDE_RE = re.compile(
    r'(?i)\b(out schedule|vacation schedule|football schedule|practice)\b'
)

# Travel / itinerary exclusion
ITINERARY_RE = re.compile(
    r"(?i)\b("
    r"itinerary|flight|flt\.?|depart|arrive|arrival|hotel|inn|reservation|"
    r"seat\s*\d+|car\s*rental|rental car|check[- ]?in|check[- ]?out|"
    r"airport|terminal"
    r")\b"
)

# Room/location tokens common in Enron meetings
ROOM_RE = re.compile(
    r'(?i)\b(eb\s*\d{2,4}[a-z]?\d*|ecn\s*\d+|ecs\s*\d+|room\s*[-\w]{2,}|three allen ctr\.?\s*-\s*\w+)\b'
)

def _strip_headerish(text: str) -> str:
    if not text:
        return ""
    return "\n".join(
        ln for ln in text.splitlines()
        if not HEADERISH_RE.search(ln.strip())
    )

def has_meeting_details_signal(subject: str, body: str) -> bool:
    subject = subject or ""
    body = body or ""

    cleaned_body = _strip_headerish(body)
    text = f"{subject}\n{cleaned_body}".lower()

    if MARKETING_RE.search(text):
        return False
    if SCHEDULE_EXCLUDE_RE.search(text):
        return False
    if ITINERARY_RE.search(text):
        return False
    if not BUSINESS_INTENT_RE.search(text):
        return False

    strong_link_patterns = [
        r"zoom\.us/j/",
        r"teams\.microsoft\.com/l/meetup-join",
        r"meet\.google\.com/",
        r"webex\.com",
        r"gotomeeting\.com",
        r"\bjoin (?:the )?meeting\b",
        r"\bjoin (?:the )?call\b",
        r"\bconference call\b",
        r"\bdial[- ]?in\b",
        r"\bmeeting id\b",
        r"\bpasscode\b",
        r"\baccess code\b",
        r"\.ics\b",
    ]

    when_where_patterns = [
        r"^\s*when\s*:\s*.+$",
        r"^\s*where\s*:\s*.+$",
        r"^\s*location\s*:\s*.+$",
        r"^\s*time\s*:\s*.+$",
        r"^\s*date\s*:\s*.+$",
        r"^\s*starts?\s*:\s*.+$",
        r"^\s*ends?\s*:\s*.+$",
    ]

    time_patterns = [
        r"\b\d{1,2}:\d{2}\s*(am|pm)\b",
        r"\b\d{1,2}\s*(am|pm)\b",
        r"\b\d{1,2}:\d{2}\b",
        r"\b\d{1,2}\.\d{2}\s*(am|pm)?\b",
        r"\b\d{1,2}:\d{2}\s*-\s*\d{1,2}:\d{2}\s*(am|pm)?\b",
    ]
    date_patterns = [
        r"\b\d{1,2}/\d{1,2}/\d{2,4}\b",
        r"\b\d{1,2}-\d{1,2}-\d{2,4}\b",
        r"\b(jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)[a-z]*\s+\d{1,2}\b",
        r"\b\d{1,2}\s+(jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)[a-z]*\b",
        r"\b(today|tomorrow)\b",
    ]

    has_strong = any(re.search(p, text) for p in strong_link_patterns)
    has_whenwhere = any(re.search(p, text, flags=re.MULTILINE) for p in when_where_patterns)
    has_room = bool(ROOM_RE.search(text))
    has_time = any(re.search(p, text) for p in time_patterns)
    has_date = any(re.search(p, text) for p in date_patterns)

    return has_strong or has_whenwhere or has_room or (has_date and has_time)


# ✅ NEW: Binary zero-shot verifier (final gate)
class MeetingIntentVerifier:
    """
    Verifies: is this email actually scheduling an event (vs just mentioning meetings)?
    """
    def __init__(self):
        self.zs = pipeline(
            "zero-shot-classification",
            model="facebook/bart-large-mnli",
            device=0 if torch.cuda.is_available() else -1
        )
        self.labels = ["meeting scheduling/invitation", "not a scheduling email"]

    def score(self, subject: str, body: str) -> Tuple[bool, float]:
        text = f"{subject}\n{body[:600]}"
        out = self.zs(text, self.labels, multi_label=False)
        pred_label = out["labels"][0]
        pred_score = float(out["scores"][0])
        return (pred_label == "meeting scheduling/invitation"), pred_score


def filter_meeting_emails(emails: List[Dict], classifier, verifier) -> List[Dict]:
    print("\nFiltering for meeting emails...")
    print(f"Processing {len(emails)} emails")

    meeting_emails = []
    seen_hashes = set()

    stats = {
        'total': len(emails),
        'quality_filtered': 0,
        'duplicates': 0,
        'meeting_found': 0,
        'detail_signal_filtered': 0,
        'low_conf_filtered': 0,
        'intent_filtered': 0,   # ✅ NEW
    }

    CONF_FLOOR = 0.80
    INTENT_FLOOR = 0.80

    for email in tqdm(emails, desc="Filtering"):
        is_quality, _ = is_quality_email(email)
        if not is_quality:
            stats['quality_filtered'] += 1
            continue

        if Config.USE_DEDUPLICATION:
            content_hash = generate_content_hash(email.get('body_clean', ''))
            if content_hash in seen_hashes:
                stats['duplicates'] += 1
                continue
            seen_hashes.add(content_hash)

        features = extract_meeting_features(
            email.get('subject_clean', ''),
            email.get('body_clean', '')
        )

        if Config.USE_ZERO_SHOT_FILTER and classifier:
            is_meeting, confidence = classifier.classify(
                email.get('subject_clean', ''),
                email.get('body_clean', '')
            )
        else:
            is_meeting = features.get('heuristic_score', 0.0) > 0.5
            confidence = features.get('heuristic_score', 0.0)

        if is_meeting:
            if confidence < CONF_FLOOR:
                stats['low_conf_filtered'] += 1
                continue

            if not has_meeting_details_signal(
                email.get('subject_clean', ''),
                email.get('body_clean', '')
            ):
                stats['detail_signal_filtered'] += 1
                continue

            # ✅ FINAL: verify true scheduling intent
            is_event, event_conf = verifier.score(
                email.get('subject_clean', ''),
                email.get('body_clean', '')
            )
            if (not is_event) or (event_conf < INTENT_FLOOR):
                stats['intent_filtered'] += 1
                continue

            email['features'] = features
            email['meeting_confidence'] = confidence
            email['event_intent_confidence'] = event_conf
            meeting_emails.append(email)
            stats['meeting_found'] += 1

    print(f"\nFiltering complete!")
    print(f"  Total processed: {stats['total']}")
    print(f"  Quality filtered: {stats['quality_filtered']}")
    print(f"  Duplicates removed: {stats['duplicates']}")
    print(f"  Low-confidence filtered (<{CONF_FLOOR}): {stats['low_conf_filtered']}")
    print(f"  Filtered by strict detail signal: {stats['detail_signal_filtered']}")
    print(f"  Filtered by intent verifier (<{INTENT_FLOOR}): {stats['intent_filtered']}")
    print(f"  Meeting emails found (final): {stats['meeting_found']}")

    return meeting_emails


# Run filter
if len(all_emails) > 0:
    verifier = MeetingIntentVerifier()
    meeting_emails = filter_meeting_emails(all_emails, classifier, verifier)
else:
    meeting_emails = []




Filtering for meeting emails...
Processing 500000 emails


Filtering: 100%|██████████| 500000/500000 [2:14:42<00:00, 61.86it/s]


Filtering complete!
  Total processed: 500000
  Quality filtered: 94240
  Duplicates removed: 224943
  Low-confidence filtered (<0.8): 104224
  Filtered by strict detail signal: 40381
  Filtered by intent verifier (<0.8): 2264
  Meeting emails found (final): 4172


In [ ]:
import pandas as pd

summary = pd.DataFrame([{
    "Total processed": 500000,
    "Quality filtered": 94240,
    "Duplicates removed": 224943,
    "Low-confidence filtered (<0.8)": 104224,
    "Detail signal filtered": 40381,
    "Intent verifier filtered (<0.8)": 2264,
    "Final meeting emails": 4172
}])

print("=== Filtering Summary ===")
display(summary)

=== Filtering Summary ===


,Total processed,Quality filtered,Duplicates removed,Low-confidence filtered (<0.8),Detail signal filtered,Intent verifier filtered (<0.8),Final meeting emails
0,500000,94240,224943,104224,40381,2264,4172


In [ ]:
def quick_meeting_sanity(meeting_emails):
    total = len(meeting_emails)
    if total == 0:
        print("No meeting_emails found.")
        return

    def has_link(e):
        t = (e.get("subject_clean","") + "\n" + e.get("body_clean","")).lower()
        return any(x in t for x in ["zoom.us/j/", "teams.microsoft.com/l/meetup-join", "meet.google.com/", "webex.com", "gotomeeting.com"])

    def has_when_where(e):
        t = (e.get("body_clean","") or "").lower()
        return any(k in t for k in ["when:", "where:", "location:", "starts:", "ends:", "date:", "time:"])

    def has_date_time_regex(e):
        t = (e.get("subject_clean","") + "\n" + e.get("body_clean","")).lower()
        date_hit = bool(re.search(r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b", t) or re.search(r"\b(jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)[a-z]*\s+\d{1,2}\b", t))
        time_hit = bool(re.search(r"\b\d{1,2}:\d{2}\s*(am|pm)?\b", t) or re.search(r"\b\d{1,2}\s*(am|pm)\b", t))
        return date_hit and time_hit

    link_ct = sum(has_link(e) for e in meeting_emails)
    whenwhere_ct = sum(has_when_where(e) for e in meeting_emails)
    dt_ct = sum(has_date_time_regex(e) for e in meeting_emails)

    print("Meeting sanity:")
    print("Total meeting_emails:", total)
    print(f"Has meeting link: {link_ct} ({link_ct/total*100:.1f}%)")
    print(f"Has When/Where-style lines: {whenwhere_ct} ({whenwhere_ct/total*100:.1f}%)")
    print(f"Has BOTH date+time regex: {dt_ct} ({dt_ct/total*100:.1f}%)")

quick_meeting_sanity(meeting_emails)

Meeting sanity:
Total meeting_emails: 4172
Has meeting link: 1 (0.0%)
Has When/Where-style lines: 956 (22.9%)
Has BOTH date+time regex: 2782 (66.7%)


In [ ]:
total = len(meeting_emails)

link_ct = 1
whenwhere_ct = 956
dt_ct = 2782

sanity_df = pd.DataFrame([{
    "Total meetings": total,
    "Has meeting link": f"{link_ct} ({link_ct/total*100:.1f}%)",
    "Has When/Where lines": f"{whenwhere_ct} ({whenwhere_ct/total*100:.1f}%)",
    "Has BOTH date+time": f"{dt_ct} ({dt_ct/total*100:.1f}%)"
}])

print("=== Meeting Sanity Metrics ===")
display(sanity_df)

=== Meeting Sanity Metrics ===


,Total meetings,Has meeting link,Has When/Where lines,Has BOTH date+time
0,4172,1 (0.0%),956 (22.9%),2782 (66.7%)


In [ ]:
import random

def review_detected_meetings(meeting_emails, n=20):
    """
    Print random meeting_emails for manual inspection.
    Helps verify if intent detection is correct.
    """
    if len(meeting_emails) == 0:
        print("No meeting_emails to review.")
        return

    print(f"\nReviewing {n} random detected meetings out of {len(meeting_emails)}")
    print("=" * 90)

    sample = random.sample(meeting_emails, min(n, len(meeting_emails)))

    for i, e in enumerate(sample, 1):
        print(f"\n--- SAMPLE {i} ---")
        print(f"Confidence: {e.get('meeting_confidence')}")
        print(f"Subject: {e.get('subject_clean')}")
        print(f"Body Preview: {e.get('body_clean')[:300]}")
        print("-" * 90)


In [ ]:
review_detected_meetings(meeting_emails, n=20)


Reviewing 20 random detected meetings out of 4172

--- SAMPLE 1 ---
Confidence: 0.961150050163269
Subject: 
Body Preview: I'm still in my conference call. Do you want to come visit later (in other 
words, need to get out?)
------------------------------------------------------------------------------------------

--- SAMPLE 2 ---
Confidence: 0.9209895730018616
Subject: staff meeting call data for thursday, july 12th
Body Preview: Folks:

Please note this week's Staff meeting & conference call scheduled as 
noted below for this Thursday:

 * Start Date/Time: 07/12/01 THU 08:30 AM PDT 
 * Conference Host: SUE MARA

HOST INFORMATION:
------------------------------------------------------------------------------------------

--- SAMPLE 3 ---
Confidence: 0.9843953251838684
Subject: staff meeting/4102
Body Preview: CALENDAR ENTRY:	APPOINTMENT

Description:
	Staff Meeting/4102

Date:		5/9/2001
Time:		9:00 AM - 11:00 AM (Central Standard Time)

Detailed Description:
--------------------------

###Samples of Meeting detection

In [ ]:
import pandas as pd
import random, os

os.makedirs(Config.OUTPUT_DIR, exist_ok=True)

sample = random.sample(meeting_emails, 10)
df_s = pd.DataFrame([{
    "subject": e.get("subject_clean",""),
    "meeting_conf": e.get("meeting_confidence", None),
    "intent_conf": e.get("event_intent_confidence", None),
    "body_preview": (e.get("body_clean","") or "")[:300].replace("\n"," ")
} for e in sample])

out_csv = os.path.join(Config.OUTPUT_DIR, "meeting_samples_10.csv")
df_s.to_csv(out_csv, index=False)
print("Saved:", out_csv)
display(df_s)

Saved: ./output/meeting_samples_10.csv


,subject,meeting_conf,intent_conf,body_preview
0,transportation model,0.978570,0.837037,---------------------- Forwarded by Vince J Ka...
1,strategy meeting,0.978628,0.839574,From: David W Delainey 10/06/2000 12:32 PM \t ...
2,are we on for lunch today,0.953992,0.929552,You bet! Can we meet at 11:20 or 11:30? I woul...
3,regularly scheduled conference call for us reg...,0.903643,0.957342,This is during our meeting with Michael Guerri...
4,200 team meeting re nw exclusivity,0.930908,0.881449,"Hi there, Just confirming a 200 CST Enron tea..."
5,ecc october meeting & training opportunity,0.918975,0.915937,"Enron Cycling Club Meeting Tuesday, October 16..."
6,regulatory meeting,0.980604,0.917570,"no, no, no. thank YOU! \tMichelle Hicks@ENRON..."
7,agenda for 8:30 am risk controls meeting (tues...,0.965701,0.855326,Agenda for tomorrow's 8:30 Risk Controls meeti...
8,whalley's budget meeting - 6/12,0.993326,0.968278,Subject:\tWhalley's Budget Meeting \t\t\tDate...
9,reminder - esa legal management meeting,0.937176,0.882683,The above meeting has been rescheduled for Thu...


In [ ]:
from google.colab import files
files.download(out_csv)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

###Meeting ang Non-meeting emails jsonl

In [ ]:
import os, json, random
from typing import List, Dict, Tuple

def _email_uid(e: Dict, fallback_prefix: str = "email") -> str:
    for k in ["msg_id", "email_id", "id", "filepath", "file_path", "path"]:
        v = e.get(k)
        if v:
            return str(v)
    subj = e.get("subject_clean", "") or e.get("subject_raw", "") or ""
    body = e.get("body_clean", "") or ""
    return f"{fallback_prefix}_{abs(hash((subj[:200], body[:1000])))}"

def save_jsonl(path: str, rows: List[Dict]) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def build_intent_jsonls(
    all_emails: List[Dict],
    meeting_emails: List[Dict],
    out_dir: str,
    max_non_meeting: int = 10000,   # ✅ cap non-meetings here
    seed: int = 42,
) -> Tuple[str, str]:
    random.seed(seed)

    # --- meeting set ---
    meeting_ids = set(_email_uid(e) for e in meeting_emails)
    meeting_rows = [{
        "email_id": _email_uid(e),
        "label": 1,
        "subject": e.get("subject_clean", "") or e.get("subject_raw", "") or "",
        "body": e.get("body_clean", "") or "",
    } for e in meeting_emails]

    # --- non-meeting pool ---
    non_pool = []
    for e in all_emails:
        ok, _ = is_quality_email(e)  # keep your quality gate
        if not ok:
            continue
        uid = _email_uid(e)
        if uid in meeting_ids:
            continue
        non_pool.append({
            "email_id": uid,
            "label": 0,
            "subject": e.get("subject_clean", "") or e.get("subject_raw", "") or "",
            "body": e.get("body_clean", "") or "",
        })

    # --- cap non-meetings to ~10K ---
    if max_non_meeting is not None and len(non_pool) > max_non_meeting:
        non_pool = random.sample(non_pool, max_non_meeting)

    # --- save ---
    meeting_path = os.path.join(out_dir, "meeting_intent.jsonl")
    non_meeting_path = os.path.join(out_dir, "non_meeting_intent.jsonl")

    save_jsonl(meeting_path, meeting_rows)
    save_jsonl(non_meeting_path, non_pool)

    print("✅ Intent JSONLs saved")
    print("  meeting_intent.jsonl     :", len(meeting_rows), "->", meeting_path)
    print("  non_meeting_intent.jsonl :", len(non_pool), "->", non_meeting_path)

    return meeting_path, non_meeting_path


# ---------- RUN ----------
intent_out_dir = os.path.join(Config.OUTPUT_DIR, "intent_data")

meeting_path, non_meeting_path = build_intent_jsonls(
    all_emails=all_emails,
    meeting_emails=meeting_emails,
    out_dir=intent_out_dir,
    max_non_meeting=10000,  # ✅ around 10K negatives
    seed=42
)


✅ Intent JSONLs saved
  meeting_intent.jsonl     : 4172 -> ./output/intent_data/meeting_intent.jsonl
  non_meeting_intent.jsonl : 10000 -> ./output/intent_data/non_meeting_intent.jsonl


In [ ]:
from google.colab import files

files.download('/content/output/intent_data/meeting_intent.jsonl')
files.download('/content/output/intent_data/non_meeting_intent.jsonl')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

###Non-meeting emails samples

In [ ]:
import json, random

def peek_jsonl(path, n=3):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    for r in random.sample(rows, n):
        print("="*90)
        print("label:", r["label"])
        print("subject:", r["subject"][:150])
        print("body:", r["body"][:250].replace("\n"," "))

peek_jsonl(non_meeting_path, n=3)

label: 0
subject: accomplishments 2000
body: SK - I've printed this out & put in the folder w/ the others. mm  Happy New Year to you and your family, Steve! I hope you got something of a  break despite all the brushfires.  Please find attached the 2000 achievements that you asked for. The only 
label: 0
subject: var
body: EFF_DT PORTFOLIO_ID DOWN95 12/15/00 MANAGEMENT-CRD 0 12/15/00 MANAGEMENT-GAS 2,262,721. 12/15/00 MANAGEMENT-PWR 343,016. 12/15/00 AGG-MANAGEMENT 2,455,841.
label: 0
subject: just thinking...
body: I was just thinking, if you could get to my office a little earlier, say 5:00  on Monday, I could take you on my "nickle" tour of Houston while it is still  light outside, if you like?!


In [ ]:
import os, json
import pandas as pd

save_dir = "/content/output/intent_data"
os.makedirs(save_dir, exist_ok=True)

meeting_path = os.path.join(save_dir, "meeting_emails_4172.jsonl")

with open(meeting_path, "w") as f:
    for e in meeting_emails:
        clean_e = e.copy()

        # Convert Timestamp to ISO string
        if isinstance(clean_e.get("date_utc"), pd.Timestamp):
            clean_e["date_utc"] = clean_e["date_utc"].isoformat()

        f.write(json.dumps(clean_e) + "\n")

print("Saved:", meeting_path)

Saved: /content/output/intent_data/meeting_emails_4172.jsonl


In [ ]:
from google.colab import files
files.download(meeting_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Sample Diverse Emails

Select a diverse set of emails across confidence levels for labeling.

In [ ]:
print("all_emails:", len(all_emails) if 'all_emails' in globals() else "MISSING")
print("meeting_emails:", len(meeting_emails) if 'meeting_emails' in globals() else "MISSING")
print("classifier:", "OK" if 'classifier' in globals() and classifier is not None else "None")

all_emails: 500000
meeting_emails: 4172
classifier: OK


In [ ]:
def prepare_emails_for_labeling(meeting_emails: List[Dict]) -> List[Dict]:
    """
    Use ALL filtered meeting emails for labeling.
    No sampling, no stratification.
    """

    print("\nPreparing emails for Gemini labeling...")

    if len(meeting_emails) == 0:
        print("ERROR: No meeting emails found")
        return []

    print(f"Total meeting_emails available: {len(meeting_emails)}")

    print(f"High confidence (>0.8): {sum(e.get('meeting_confidence', 0) > 0.8 for e in meeting_emails)}")
    print(f"Medium confidence (0.65–0.8): {sum(0.65 < e.get('meeting_confidence', 0) <= 0.8 for e in meeting_emails)}")
    print(f"Borderline (0.5–0.65): {sum(0.5 <= e.get('meeting_confidence', 0) <= 0.65 for e in meeting_emails)}")

    print("\nUsing ALL emails for labeling.")
    return meeting_emails


# Use all emails
emails_to_label = prepare_emails_for_labeling(meeting_emails)


Preparing emails for Gemini labeling...
Total meeting_emails available: 4172
High confidence (>0.8): 4172
Medium confidence (0.65–0.8): 0
Borderline (0.5–0.65): 0

Using ALL emails for labeling.


In [ ]:
# Label EVERYTHING we detected as meeting emails
emails_for_labeling = meeting_emails
print("✅ Emails to label with Gemini:", len(emails_for_labeling))

✅ Emails to label with Gemini: 4172


---

## PART 5: GEMINI LABELING

Use Gemini API to generate ground truth labels for meeting extraction.

**Note:** This section uses the Gemini API and will incur costs (~$2-3 for 500 emails).

In [ ]:
import google.generativeai as genai

genai.configure(api_key=Config.GEMINI_API_KEY)

models = list(genai.list_models())
print("Total models:", len(models))

for m in models:
    methods = getattr(m, "supported_generation_methods", [])
    if "generateContent" in methods:
        print(m.name, "->", methods)

In [ ]:
import google.generativeai as genai
genai.configure(api_key=Config.GEMINI_API_KEY)

model = genai.GenerativeModel("models/gemini-2.5-flash")
resp = model.generate_content("Reply ONLY with: OK")
print(resp.text)

OK


### Define Labeling Functions

In [ ]:
import re
import json
from typing import Dict, Tuple, Optional

def create_labeling_prompt(email_dict: Dict) -> str:
    """
    Create prompt for Gemini to extract meeting details.
    """
    email_text = f"Subject: {email_dict.get('subject_raw','')}\n\n{email_dict.get('body_clean','')}"

    # Build attendee list from headers
    all_attendees = [email_dict.get('from_email')] + email_dict.get('to_emails', []) + email_dict.get('cc_emails', [])
    all_attendees = [a for a in all_attendees if a]

    prompt = f"""You are a meeting information extraction system. Extract meeting details from the following email.

Email:
---
{email_text}
---

Extract the following information in JSON format:
{{
  "title": "<meeting title or subject>",
  "attendees": ["<email1@domain.com>", "<email2@domain.com>"],
  "start_utc": "<ISO 8601 format: YYYY-MM-DDTHH:MM:SSZ or null>",
  "end_utc": "<ISO 8601 format: YYYY-MM-DDTHH:MM:SSZ or null>",
  "location": "<meeting location or null>"
}}

Rules:
1. If no specific time is mentioned, use null for start_utc and end_utc
2. Extract all email addresses mentioned as potential attendees
3. Include these addresses if relevant: {', '.join(all_attendees[:5])}
4. Convert all times to UTC in ISO 8601 format (assume PST/PDT timezone if not specified)
5. Extract location if explicitly mentioned (room names, addresses, virtual links)
6. For title, use the meeting subject or a brief description from the email body
7. Return ONLY a valid JSON object.
   Do NOT include markdown, explanations, or any text before/after JSON.

JSON:"""

    return prompt


def extract_json_from_response(response_text: str) -> str:
    """
    Extract JSON from Gemini response, handling markdown formatting.
    """
    response_text = (response_text or "").strip()

    # Remove markdown code blocks if present
    if response_text.startswith('```'):
        response_text = re.sub(r'^```json\s*', '', response_text, flags=re.IGNORECASE)
        response_text = re.sub(r'^```\s*', '', response_text)
        response_text = re.sub(r'```\s*$', '', response_text)

    return response_text.strip()


def label_email_with_gemini(email_dict: Dict, model) -> Tuple[Optional[Dict], Optional[str]]:
    """
    Label email with Gemini API.

    Returns:
        (labels_dict, error_message)
    """
    try:
        prompt = create_labeling_prompt(email_dict)

        # ✅ deterministic output for structured extraction
        response = model.generate_content(
            prompt,
            generation_config={"temperature": 0.0}
        )

        # ✅ guard against empty/blocked responses
        if (response is None) or (not getattr(response, "text", None)):
            return None, "empty_response"

        response_text = extract_json_from_response(response.text)

        labels = json.loads(response_text)

        # Validate schema
        required_fields = ['title', 'attendees', 'start_utc', 'end_utc', 'location']
        missing = [f for f in required_fields if f not in labels]
        if missing:
            return None, f"missing_fields: {missing}"

        # Validate types
        if not isinstance(labels['attendees'], list):
            return None, "attendees_not_list"

        return labels, None

    except json.JSONDecodeError as e:
        return None, f"json_decode_error: {str(e)}"
    except Exception as e:
        return None, str(e)

print("Gemini labeling functions defined (modified)")

Gemini labeling functions defined (modified)


### Run Gemini Labeling

In [ ]:
import time
import google.generativeai as genai
from tqdm import tqdm
from typing import List, Dict

def label_with_gemini(emails_to_label: List[Dict]) -> List[Dict]:
    """
    Label all meeting emails with Gemini.
    """

    print("\nLabeling emails with Gemini...")
    print(f"Total emails to label: {len(emails_to_label)}")

    # Rough cost estimate (flash model is very cheap)
    print(f"Estimated cost (very rough): ${len(emails_to_label) * 0.002:.2f}")

    # Validate API key
    if not Config.GEMINI_API_KEY or Config.GEMINI_API_KEY.startswith("YOUR_"):
        print("ERROR: GEMINI_API_KEY not properly set in Config.")
        return []

    # Initialize Gemini
    genai.configure(api_key=Config.GEMINI_API_KEY)
    model = genai.GenerativeModel("models/gemini-2.5-flash")

    labeled_emails = []
    failed_count = 0

    for i, email in enumerate(tqdm(emails_to_label, desc="Labeling")):
        labels, error = label_email_with_gemini(email, model)

        if labels:
            labeled_email = {
                'email_id': email.get('msg_id', email.get('email_id', f"email_{i}")),
                'filepath': email.get('filepath', email.get('file_path', email.get('path'))),
                'from_email': email.get('from_email'),
                'to_emails': email.get('to_emails', []),
                'cc_emails': email.get('cc_emails', []),
                'subject_raw': email.get('subject_raw', ''),
                'subject_clean': email.get('subject_clean', ''),
                'body_clean': email.get('body_clean', ''),
                'date_utc': str(email.get('date_utc', '')),
                'features': email.get('features', {}),
                'meeting_confidence': email.get('meeting_confidence'),
                'labels': labels,
            }

            labeled_emails.append(labeled_email)
        else:
            failed_count += 1

        # Gentle rate limiting
        if (i + 1) % 15 == 0:
            time.sleep(1)

    print("\nLabeling complete!")
    print(f"  Successfully labeled: {len(labeled_emails)}")
    print(f"  Failed: {failed_count}")
    print(f"  Success rate: {len(labeled_emails) / len(emails_to_label) * 100:.2f}%")

    return labeled_emails


# ✅ Run labeling on ALL 1495
if len(meeting_emails) > 0:
    labeled_emails = label_with_gemini(meeting_emails)
else:
    print("ERROR: No emails to label")
    labeled_emails = []





Labeling emails with Gemini...
Total emails to label: 4172
Estimated cost (very rough): $8.34



Labeling:   0%|          | 0/4172 [01:11<?, ?it/s]


KeyboardInterrupt: 

###Do not run this

In [ ]:
# ============================================================
# FAST + SAFE GEMINI LABELING (checkpoint + resume + batching)
# For 4,172 emails on Colab (A100 doesn’t speed API, batching does)
# ============================================================

import os, json, time, random
from typing import List, Dict, Optional, Tuple
from tqdm import tqdm
import google.generativeai as genai

# ---------------------------
# Helpers: JSONL checkpointing
# ---------------------------
def _ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def _append_jsonl(path: str, obj: Dict) -> None:
    _ensure_dir(os.path.dirname(path))
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def _load_done_ids(jsonl_path: str) -> set:
    done = set()
    if not os.path.exists(jsonl_path):
        return done
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                eid = obj.get("email_id")
                if eid is not None:
                    done.add(str(eid))
            except:
                pass
    return done

def _email_id(e: Dict, i: int) -> str:
    return str(e.get("msg_id") or e.get("email_id") or e.get("id") or f"email_{i}")

def _is_transient(err: str) -> bool:
    if not err:
        return False
    s = err.lower()
    return any(k in s for k in [
        "429", "rate", "quota", "timeout", "temporarily", "unavailable", "503", "internal", "resource"
    ])

# ---------------------------
# Batch call wrapper (works across SDK variants)
# ---------------------------
def _batch_generate(model_name: str, prompts: List[str], temperature: float = 0.0):
    """
    Tries multiple batch APIs depending on installed google-generativeai SDK version.
    Returns a list of response-like objects (each should have .text or be convertible).
    """
    # Variant A: genai.batch_generate_content(...)
    if hasattr(genai, "batch_generate_content"):
        try:
            return genai.batch_generate_content(
                model=model_name,
                requests=[
                    {
                        # Common request shape accepted by many versions:
                        "contents": p,
                        "generation_config": {"temperature": temperature},
                    } for p in prompts
                ],
            )
        except Exception:
            pass

        # Try stricter contents format
        try:
            return genai.batch_generate_content(
                model=model_name,
                requests=[
                    {
                        "contents": [{"role": "user", "parts": [{"text": p}]}],
                        "generation_config": {"temperature": temperature},
                    } for p in prompts
                ],
            )
        except Exception:
            pass

    # Variant B: model.batch_generate_content(...)
    try:
        m = genai.GenerativeModel(model_name)
        if hasattr(m, "batch_generate_content"):
            try:
                return m.batch_generate_content(
                    requests=[
                        {"contents": p, "generation_config": {"temperature": temperature}}
                        for p in prompts
                    ]
                )
            except Exception:
                return m.batch_generate_content(
                    requests=[
                        {"contents": [{"role": "user", "parts": [{"text": p}]}],
                         "generation_config": {"temperature": temperature}}
                        for p in prompts
                    ]
                )
    except Exception:
        pass

    raise RuntimeError("Batch API not supported by this SDK/model. Fallback to single calls.")

# ---------------------------
# Main: checkpointed labeling
# ---------------------------
def label_with_gemini_checkpointed(
    emails_to_label: List[Dict],
    model_name: str = "models/gemini-2.5-flash",
    out_dir: Optional[str] = None,
    out_name: str = "gemini_meeting_labels.jsonl",
    batch_size: int = 8,              # try 8; if you see rate errors, reduce to 4
    temperature: float = 0.0,
    base_sleep: float = 0.10,         # small throttle
    jitter: float = 0.15,
    max_retries: int = 4,
    retry_base: float = 0.8,
    use_batch: bool = True,
) -> str:
    """
    Labels emails with Gemini and writes output incrementally to JSONL.
    - resumes from existing JSONL (skips done IDs)
    - writes success rows + error rows (so you can retry only failures)
    """
    if out_dir is None:
        out_dir = os.path.join(Config.OUTPUT_DIR, "gemini_labels")
    _ensure_dir(out_dir)
    out_path = os.path.join(out_dir, out_name)

    # Configure Gemini
    if not getattr(Config, "GEMINI_API_KEY", None) or str(Config.GEMINI_API_KEY).startswith("YOUR_"):
        raise ValueError("Config.GEMINI_API_KEY not set properly.")
    genai.configure(api_key=Config.GEMINI_API_KEY)

    # For single-call fallback
    single_model = genai.GenerativeModel(model_name)

    done_ids = _load_done_ids(out_path)
    print("Output JSONL:", out_path)
    print("Total provided:", len(emails_to_label))
    print("Already done  :", len(done_ids))

    queue: List[Tuple[str, Dict]] = []
    for i, e in enumerate(emails_to_label):
        eid = _email_id(e, i)
        if eid not in done_ids:
            queue.append((eid, e))

    print("Remaining     :", len(queue))
    if not queue:
        print("✅ Nothing to do.")
        return out_path

    ok_ct = 0
    fail_ct = 0

    def _label_one(eid: str, email: Dict) -> Tuple[Optional[Dict], Optional[str]]:
        # Uses your existing function:
        # label_email_with_gemini(email_dict, model) -> (labels_dict or None, error or None)
        return label_email_with_gemini(email, single_model)

    pbar = tqdm(total=len(queue), desc="Gemini labeling")

    idx = 0
    while idx < len(queue):
        time.sleep(base_sleep + random.random() * jitter)

        # -----------------------
        # Batch path (faster)
        # -----------------------
        if use_batch:
            batch = queue[idx: idx + batch_size]
            idx += len(batch)

            prompts = [create_labeling_prompt(e) for _, e in batch]

            batch_responses = None
            batch_failed = False
            try:
                batch_responses = _batch_generate(model_name, prompts, temperature=temperature)
            except Exception as ex:
                batch_failed = True
                batch_err = str(ex)

            for j, (eid, email) in enumerate(batch):
                # If batch failed entirely, fallback to single for each
                if batch_failed:
                    labels, err = _label_one(eid, email)
                else:
                    # Try to read each response's text in a robust way
                    try:
                        r = batch_responses[j]
                        txt = getattr(r, "text", None)
                        if not txt and isinstance(r, dict):
                            txt = r.get("text")
                        if not txt:
                            # last resort: stringify
                            txt = str(r)

                        cleaned = extract_json_from_response(txt)
                        labels = json.loads(cleaned)

                        required = ["title", "attendees", "start_utc", "end_utc", "location"]
                        missing = [f for f in required if f not in labels]
                        if missing:
                            raise ValueError(f"missing_fields: {missing}")
                        if not isinstance(labels.get("attendees"), list):
                            raise ValueError("attendees_not_list")

                        err = None
                    except Exception as ex:
                        labels, err = None, str(ex)

                # Retry transient errors (single-call retries)
                if labels is None and _is_transient(err):
                    last_err = err
                    for rtry in range(max_retries):
                        time.sleep((2 ** rtry) * retry_base + random.random() * 0.2)
                        labels2, err2 = _label_one(eid, email)
                        if labels2 is not None:
                            labels, err = labels2, None
                            break
                        last_err = err2
                    if labels is None:
                        err = last_err

                if labels is not None:
                    row = {
                        "email_id": eid,
                        "filepath": email.get("filepath", email.get("file_path", email.get("path"))),
                        "from_email": email.get("from_email"),
                        "to_emails": email.get("to_emails", []),
                        "cc_emails": email.get("cc_emails", []),
                        "subject_raw": email.get("subject_raw", ""),
                        "subject_clean": email.get("subject_clean", ""),
                        "body_clean": email.get("body_clean", ""),
                        "date_utc": str(email.get("date_utc", "")),
                        "features": email.get("features", {}),
                        "meeting_confidence": email.get("meeting_confidence"),
                        "event_intent_confidence": email.get("event_intent_confidence"),
                        "labels": labels,
                    }
                    _append_jsonl(out_path, row)
                    ok_ct += 1
                else:
                    _append_jsonl(out_path, {"email_id": eid, "error": err or "unknown_error"})
                    fail_ct += 1

                pbar.update(1)
                pbar.set_postfix({"ok": ok_ct, "fail": fail_ct})

        # -----------------------
        # Single path (slower)
        # -----------------------
        else:
            eid, email = queue[idx]
            idx += 1

            labels, err = _label_one(eid, email)

            if labels is None and _is_transient(err):
                last_err = err
                for rtry in range(max_retries):
                    time.sleep((2 ** rtry) * retry_base + random.random() * 0.2)
                    labels2, err2 = _label_one(eid, email)
                    if labels2 is not None:
                        labels, err = labels2, None
                        break
                    last_err = err2
                if labels is None:
                    err = last_err

            if labels is not None:
                row = {
                    "email_id": eid,
                    "filepath": email.get("filepath", email.get("file_path", email.get("path"))),
                    "from_email": email.get("from_email"),
                    "to_emails": email.get("to_emails", []),
                    "cc_emails": email.get("cc_emails", []),
                    "subject_raw": email.get("subject_raw", ""),
                    "subject_clean": email.get("subject_clean", ""),
                    "body_clean": email.get("body_clean", ""),
                    "date_utc": str(email.get("date_utc", "")),
                    "features": email.get("features", {}),
                    "meeting_confidence": email.get("meeting_confidence"),
                    "event_intent_confidence": email.get("event_intent_confidence"),
                    "labels": labels,
                }
                _append_jsonl(out_path, row)
                ok_ct += 1
            else:
                _append_jsonl(out_path, {"email_id": eid, "error": err or "unknown_error"})
                fail_ct += 1

            pbar.update(1)
            pbar.set_postfix({"ok": ok_ct, "fail": fail_ct})

    pbar.close()
    print("\n✅ Completed labeling.")
    print("Saved to:", out_path)
    print("Success :", ok_ct)
    print("Failed  :", fail_ct)
    return out_path

# ---------------------------
# RUN IT (recommended settings)
# ---------------------------
out_path = label_with_gemini_checkpointed(
    emails_to_label=meeting_emails,  # <-- 4172
    model_name="models/gemini-2.5-flash",
    out_dir=os.path.join(Config.OUTPUT_DIR, "gemini_labels"),
    out_name="gemini_meeting_labels.jsonl",
    batch_size=8,       # if rate-limit errors, set to 4
    use_batch=True,     # batching is the speed win
    temperature=0.0,
    base_sleep=0.10
)
print("✅ Output JSONL:", out_path)

Output JSONL: ./output/gemini_labels/gemini_meeting_labels.jsonl
Total provided: 4172
Already done  : 0
Remaining     : 4172


Gemini labeling:   0%|          | 0/4172 [00:00<?, ?it/s]

KeyboardInterrupt: 

### Validate and Save Labeled Data

In [ ]:
# Validate label quality
if len(labeled_emails) > 0:
    print("\nLabel Statistics:")
    print("=" * 80)

    has_title = sum(1 for e in labeled_emails if e['labels']['title'])
    has_attendees = sum(1 for e in labeled_emails if e['labels']['attendees'])
    has_start_time = sum(1 for e in labeled_emails if e['labels']['start_utc'])
    has_end_time = sum(1 for e in labeled_emails if e['labels']['end_utc'])
    has_location = sum(1 for e in labeled_emails if e['labels']['location'])

    total = len(labeled_emails)
    print(f"Total labeled: {total}")
    print(f"Has title: {has_title} ({has_title/total*100:.1f}%)")
    print(f"Has attendees: {has_attendees} ({has_attendees/total*100:.1f}%)")
    print(f"Has start time: {has_start_time} ({has_start_time/total*100:.1f}%)")
    print(f"Has end time: {has_end_time} ({has_end_time/total*100:.1f}%)")
    print(f"Has location: {has_location} ({has_location/total*100:.1f}%)")

    avg_attendees = np.mean([len(e['labels']['attendees']) for e in labeled_emails])
    print(f"Average attendees: {avg_attendees:.1f}")

    # Display sample
    print("\nSample labeled email:")
    print("=" * 80)
    sample = labeled_emails[0]
    print(f"Subject: {sample['subject_raw']}")
    print(f"Body: {sample['body_clean'][:200]}...")
    print(f"\nLabels:")
    print(json.dumps(sample['labels'], indent=2))
    print("=" * 80)

    # Save labeled data
    output_file = f"{Config.OUTPUT_DIR}/labeled_emails_{Config.SAMPLE_SIZE}.jsonl"
    with open(output_file, 'w') as f:
        for email in labeled_emails:
            f.write(json.dumps(email, default=str) + '\n')

    print(f"\nSaved labeled data to: {output_file}")
else:
    print("ERROR: No labeled emails to save")


Label Statistics:
Total labeled: 499
Has title: 499 (100.0%)
Has attendees: 499 (100.0%)
Has start time: 79 (15.8%)
Has end time: 50 (10.0%)
Has location: 60 (12.0%)
Average attendees: 5.3

Sample labeled email:
Subject: 
Body: Hello:
I am not able to pull up the link for the short term outlook for natural 
gas. Can you please make sure the link is updates....

Labels:
{
  "title": "Short Term Outlook for Natural Gas",
  "attendees": [
    "john.arnold@enron.com",
    "kendrick.brown@eia.doe.gov"
  ],
  "start_utc": null,
  "end_utc": null,
  "location": null
}

Saved labeled data to: ./output/labeled_emails_500.jsonl


---

## PART 6: FINE-TUNING PREPARATION

Prepare data for fine-tuning: split into train/val/test and format for instruction tuning.

### Create Train/Val/Test Splits

In [ ]:
def create_train_val_test_splits(labeled_emails: List[Dict]) -> Tuple[List[Dict], List[Dict], List[Dict]]:
    """
    Split labeled data into train/validation/test sets.

    Args:
        labeled_emails: List of labeled emails

    Returns:
        Tuple of (train_data, val_data, test_data)
    """
    print("\nCreating train/validation/test splits...")

    # First split: train vs temp (90% / 10%)
    train_data, temp_data = train_test_split(
        labeled_emails,
        test_size=(1 - Config.TRAIN_RATIO),
        random_state=42
    )

    # Second split: val vs test from temp (50% / 50%)
    val_data, test_data = train_test_split(
        temp_data,
        test_size=0.5,
        random_state=42
    )

    print(f"Splits created:")
    print(f"  Train: {len(train_data)} samples ({Config.TRAIN_RATIO*100}%)")
    print(f"  Validation: {len(val_data)} samples ({Config.VAL_RATIO*100}%)")
    print(f"  Test: {len(test_data)} samples ({Config.TEST_RATIO*100}%)")

    return train_data, val_data, test_data

# Create splits
if len(labeled_emails) > 0:
    train_data, val_data, test_data = create_train_val_test_splits(labeled_emails)
else:
    print("ERROR: No labeled data for splitting")
    train_data, val_data, test_data = [], [], []

### Format Data for Fine-tuning

Format using Llama 3 chat template (will be applied by tokenizer later).

In [ ]:
def format_for_training(labeled_email: Dict) -> Dict:
    """
    Format labeled email for instruction-following fine-tuning.
    Creates messages dict for apply_chat_template.

    Args:
        labeled_email: Labeled email dictionary

    Returns:
        Formatted dictionary with 'messages' field
    """
    # Create input text
    email_text = f"Subject: {labeled_email['subject_raw']}\n\n{labeled_email['body_clean']}"

    # Create output JSON
    output_json = json.dumps(labeled_email['labels'])

    # System prompt
    system_prompt = "You are a meeting extraction assistant. Extract meeting details from emails and return JSON with: title, attendees, start_utc, end_utc, location."

    # Format as messages (tokenizer will apply chat template)
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Extract meeting details from the following email and return them in JSON format with fields: title, attendees, start_utc, end_utc, location.\n\n{email_text}"},
        {"role": "assistant", "content": output_json}
    ]

    return {
        "messages": messages,
        "email_id": labeled_email.get('email_id'),
    }

# Format all datasets
if len(train_data) > 0:
    print("\nFormatting data for fine-tuning...")

    train_formatted = [format_for_training(email) for email in train_data]
    val_formatted = [format_for_training(email) for email in val_data]
    test_formatted = [format_for_training(email) for email in test_data]

    # Convert to Hugging Face datasets
    train_dataset = Dataset.from_list(train_formatted)
    val_dataset = Dataset.from_list(val_formatted)
    test_dataset = Dataset.from_list(test_formatted)

    print(f"Datasets created:")
    print(f"  Train: {len(train_dataset)} samples")
    print(f"  Validation: {len(val_dataset)} samples")
    print(f"  Test: {len(test_dataset)} samples")

    # Save test data separately for evaluation
    test_file = f"{Config.FINETUNING_OUTPUT_DIR}/test_data.jsonl"
    with open(test_file, 'w') as f:
        for item in test_data:
            f.write(json.dumps(item, default=str) + '\n')
    print(f"Saved test data to: {test_file}")
else:
    print("ERROR: No training data to format")

---

## PART 7: MODEL SETUP AND FINE-TUNING

Load base model from Hugging Face, apply QLoRA, and fine-tune on our data.

**Note:** This section will download the Llama model (~6GB) and train it (~30-60 minutes on GPU).

### Setup Model with QLoRA

This cell loads the base model from Hugging Face and applies 4-bit quantization with LoRA adapters.

In [ ]:
def setup_model_and_tokenizer():
    """
    Setup model with QLoRA and tokenizer from Hugging Face.
    Uses 4-bit quantization for memory efficiency.

    Returns:
        Tuple of (model, tokenizer)
    """
    print(f"\nLoading base model: {Config.BASE_MODEL}")
    print("This may take 2-5 minutes (downloading ~6GB model)...")

    # Verify HF authentication
    if Config.HF_TOKEN == "YOUR_HUGGINGFACE_TOKEN_HERE":
        print("ERROR: HF_TOKEN not set")
        print("Please set your Hugging Face token in the Config class")
        raise ValueError("HF_TOKEN not set")

    # Quantization config for 4-bit QLoRA
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    # Load model from Hugging Face
    try:
        model = AutoModelForCausalLM.from_pretrained(
            Config.BASE_MODEL,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
            torch_dtype=torch.float16,
            use_auth_token=Config.HF_TOKEN
        )
    except Exception as e:
        print(f"Failed to load model: {e}")
        print("Make sure you've accepted the Llama license at:")
        print(f"https://huggingface.co/{Config.BASE_MODEL}")
        raise

    # Load tokenizer from Hugging Face
    tokenizer = AutoTokenizer.from_pretrained(
        Config.BASE_MODEL,
        trust_remote_code=True,
        use_auth_token=Config.HF_TOKEN,
        use_fast=True
    )

    # Set padding token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # Prepare model for k-bit training
    model = prepare_model_for_kbit_training(model)

    # Configure LoRA
    lora_config = LoraConfig(
        r=Config.LORA_R,
        lora_alpha=Config.LORA_ALPHA,
        target_modules=Config.TARGET_MODULES,
        lora_dropout=Config.LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM"
    )

    # Apply LoRA
    model = get_peft_model(model, lora_config)

    # Print trainable parameters
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    all_params = sum(p.numel() for p in model.parameters())

    print(f"\nModel loaded successfully from Hugging Face!")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"All parameters: {all_params:,}")
    print(f"Trainable percentage: {100 * trainable_params / all_params:.2f}%")
    print(f"Device map: {model.hf_device_map}")

    return model, tokenizer

# Load model and tokenizer
if len(train_data) > 0:
    model, tokenizer = setup_model_and_tokenizer()
else:
    print("ERROR: Cannot load model without training data")

### Configure Training Arguments and Start Fine-tuning

In [ ]:
if len(train_data) > 0:
    # Training configuration
    training_args = TrainingArguments(
        output_dir=Config.FINETUNING_OUTPUT_DIR,
        num_train_epochs=Config.NUM_EPOCHS,
        per_device_train_batch_size=Config.BATCH_SIZE,
        per_device_eval_batch_size=Config.BATCH_SIZE,
        gradient_accumulation_steps=Config.GRADIENT_ACCUMULATION_STEPS,
        learning_rate=Config.LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=Config.WARMUP_RATIO,
        logging_steps=Config.LOGGING_STEPS,
        evaluation_strategy="steps",
        eval_steps=Config.EVAL_STEPS,
        save_strategy="steps",
        save_steps=Config.SAVE_STEPS,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="loss",
        greater_is_better=False,
        fp16=True,
        report_to="none",
        remove_unused_columns=False,
    )

    print("\nTraining configuration:")
    print(f"  Epochs: {training_args.num_train_epochs}")
    print(f"  Batch size: {training_args.per_device_train_batch_size}")
    print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
    print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
    print(f"  Learning rate: {training_args.learning_rate}")

    # Format training function
    def formatting_func(examples):
        """Format examples using tokenizer's chat template."""
        texts = []
        for messages in examples["messages"]:
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            texts.append(text)
        return {"text": texts}

    # Apply formatting
    train_dataset = train_dataset.map(formatting_func, batched=True)
    val_dataset = val_dataset.map(formatting_func, batched=True)

    # Initialize trainer
    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        dataset_text_field="text",
        max_seq_length=Config.MAX_SEQ_LENGTH,
        packing=False,
    )

    print("\nTrainer initialized")
    print("Starting training...")
    print("This will take 30-60 minutes for 500 samples on GPU")
    print("=" * 80)

    # Train the model
    trainer.train()

    print("\n" + "=" * 80)
    print("Training complete!")

    # Save final model
    final_model_path = f"{Config.MODEL_DIR}/final_model"
    trainer.save_model(final_model_path)
    tokenizer.save_pretrained(final_model_path)

    print(f"Model saved to: {final_model_path}")
else:
    print("ERROR: Cannot train without data")

---

## PART 8: EVALUATION

Evaluate the fine-tuned model on the test set using proper chat template.

### Define Evaluation Functions with Chat Template

In [ ]:
def generate_prediction(email_text: str, model, tokenizer) -> Optional[Dict]:
    """
    Generate meeting extraction prediction for a given email using chat template.

    Args:
        email_text: Email text
        model: Trained model
        tokenizer: Tokenizer

    Returns:
        Predicted labels dictionary or None if parsing fails
    """
    system_prompt = "You are a meeting extraction assistant. Extract meeting details from emails and return JSON with: title, attendees, start_utc, end_utc, location."

    # Create messages using proper chat template
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Extract meeting details from the following email and return them in JSON format with fields: title, attendees, start_utc, end_utc, location.\n\n{email_text}"}
    ]

    # Apply chat template
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the generated part (skip input)
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

    # Extract JSON from response
    try:
        json_start = response.find('{')
        json_end = response.rfind('}') + 1

        if json_start == -1 or json_end == 0:
            return None

        json_str = response[json_start:json_end]
        prediction = json.loads(json_str)

        # Validate schema
        required_fields = ['title', 'attendees', 'start_utc', 'end_utc', 'location']
        if all(field in prediction for field in required_fields):
            return prediction
        else:
            return None

    except Exception as e:
        return None


def calculate_attendee_metrics(pred_attendees: List[str], true_attendees: List[str]) -> Tuple[float, float, float]:
    """
    Calculate precision, recall, F1 for attendee extraction.

    Args:
        pred_attendees: Predicted attendee list
        true_attendees: Ground truth attendee list

    Returns:
        Tuple of (precision, recall, f1)
    """
    if not pred_attendees and not true_attendees:
        return 1.0, 1.0, 1.0
    if not pred_attendees or not true_attendees:
        return 0.0, 0.0, 0.0

    pred_set = set(pred_attendees)
    true_set = set(true_attendees)

    true_positives = len(pred_set & true_set)
    precision = true_positives / len(pred_set) if pred_set else 0
    recall = true_positives / len(true_set) if true_set else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return precision, recall, f1

print("Evaluation functions defined")

### Run Comprehensive Evaluation

In [ ]:
if len(test_data) > 0:
    print("\nRunning evaluation on test set...")
    print(f"Evaluating {len(test_data)} test samples")
    print("This may take 5-10 minutes")

    # Metrics tracking
    json_valid = 0
    schema_compliant = 0
    exact_match = 0

    field_exact_match = {
        'title': 0,
        'location': 0,
        'start_utc': 0,
        'end_utc': 0
    }

    attendee_metrics = []

    # Evaluate each test sample
    for sample in tqdm(test_data, desc="Evaluating"):
        email_text = f"Subject: {sample['subject_raw']}\n\n{sample['body_clean']}"
        ground_truth = sample['labels']

        prediction = generate_prediction(email_text, model, tokenizer)

        # JSON validity
        if prediction is not None:
            json_valid += 1

            # Schema compliance
            required_fields = ['title', 'attendees', 'start_utc', 'end_utc', 'location']
            if all(field in prediction for field in required_fields):
                schema_compliant += 1

                # Field-level exact match
                if prediction['title'] == ground_truth['title']:
                    field_exact_match['title'] += 1
                if prediction['location'] == ground_truth['location']:
                    field_exact_match['location'] += 1
                if prediction['start_utc'] == ground_truth['start_utc']:
                    field_exact_match['start_utc'] += 1
                if prediction['end_utc'] == ground_truth['end_utc']:
                    field_exact_match['end_utc'] += 1

                # Attendee metrics
                p, r, f1 = calculate_attendee_metrics(
                    prediction.get('attendees', []),
                    ground_truth['attendees']
                )
                attendee_metrics.append({'precision': p, 'recall': r, 'f1': f1})

                # Overall exact match
                if prediction == ground_truth:
                    exact_match += 1

    # Calculate and display results
    total = len(test_data)

    print("\n" + "=" * 80)
    print("EVALUATION RESULTS")
    print("=" * 80)
    print(f"\nTest samples: {total}")
    print(f"\n1. JSON Validity: {json_valid/total*100:.2f}% ({json_valid}/{total})")

    if json_valid > 0:
        print(f"2. Schema Compliance: {schema_compliant/json_valid*100:.2f}% ({schema_compliant}/{json_valid})")

    if schema_compliant > 0:
        print(f"3. Overall Exact Match: {exact_match/schema_compliant*100:.2f}% ({exact_match}/{schema_compliant})")

        print(f"\nField-Level Exact Match:")
        for field, count in field_exact_match.items():
            print(f"  {field}: {count/schema_compliant*100:.2f}%")

        if attendee_metrics:
            avg_precision = np.mean([m['precision'] for m in attendee_metrics])
            avg_recall = np.mean([m['recall'] for m in attendee_metrics])
            avg_f1 = np.mean([m['f1'] for m in attendee_metrics])

            print(f"\nAttendee Extraction (Key Metric):")
            print(f"  Precision: {avg_precision*100:.2f}%")
            print(f"  Recall: {avg_recall*100:.2f}%")
            print(f"  F1-Score: {avg_f1*100:.2f}%")

    print("\n" + "=" * 80)

    # Save results
    results = {
        'total_samples': total,
        'json_validity': json_valid / total if total > 0 else 0,
        'schema_compliance': schema_compliant / json_valid if json_valid > 0 else 0,
        'exact_match': exact_match / schema_compliant if schema_compliant > 0 else 0,
        'field_exact_match': {k: v/schema_compliant for k, v in field_exact_match.items()} if schema_compliant > 0 else {},
        'attendee_precision': float(avg_precision) if attendee_metrics else 0,
        'attendee_recall': float(avg_recall) if attendee_metrics else 0,
        'attendee_f1': float(avg_f1) if attendee_metrics else 0,
    }

    results_file = f"{Config.FINETUNING_OUTPUT_DIR}/evaluation_results.json"
    with open(results_file, 'w') as f:
        json.dump(results, f, indent=2)

    print(f"\nEvaluation results saved to: {results_file}")
else:
    print("ERROR: No test data for evaluation")

---

## PIPELINE COMPLETE

Congratulations! You have successfully:

1. Downloaded and parsed Enron email dataset
2. Filtered for meeting-related emails using zero-shot classification
3. Labeled emails with Gemini API
4. Created train/val/test splits
5. Fine-tuned Llama 3.2 3B with QLoRA from Hugging Face
6. Evaluated on test set

**Your trained model is saved at:** `./trained_model/final_model/`

**Next steps:**
- Review evaluation results in `./finetuning_output/evaluation_results.json`
- Test your model on new emails
- Scale to 50K samples by changing `Config.SAMPLE_SIZE = 50000` and rerunning
- Deploy your model to production

### Test Your Model on a New Email

Try your trained model on a custom email:

In [ ]:
# Test on a custom email
test_email = """Subject: Weekly Team Meeting

Hi everyone,

Let's have our weekly sync tomorrow (Friday) at 10am in Conference Room 3.

Agenda:
- Project updates
- Budget review
- Q&A

Please confirm your attendance.

Thanks,
John
"""

if len(test_data) > 0:
    print("Testing model on custom email...")
    print("=" * 80)
    print(test_email)
    print("=" * 80)

    prediction = generate_prediction(test_email, model, tokenizer)

    if prediction:
        print("\nPredicted meeting details:")
        print(json.dumps(prediction, indent=2))
    else:
        print("\nFailed to extract meeting details")
else:
    print("Model not trained yet")